# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Citation:** Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.

- Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- License: [Open Data Commons Attribution License](https://opendatacommons.org/licenses/by/1-0/)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'Unknown')}")
print(f"Version: {getattr(metadata, 'version', 'Unknown')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'Unknown')}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields as specified in the Croissant schema.

In [ ]:
# List all available record sets in the dataset
print("Available Record Sets (by @id):\n")
record_sets = []
for record_set in metadata.record_sets:
    print(f"@id: {record_set.id}  |  name: {record_set.name if hasattr(record_set, 'name') else 'No name'}")
    record_sets.append(record_set.id)

# Display fields for each record set
for record_set in metadata.record_sets:
    print(f"\nRecordSet: {record_set.name if hasattr(record_set, 'name') else record_set.id}")
    print(f"  @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        dtype = getattr(field, 'data_type', 'Unknown')
        print(f"    @id: {field.id} | name: {field.name if hasattr(field, 'name') else 'No name'} | type: {dtype}")

## Preview Records
Let's show a few example records from the first available record set. All data access uses the record set `@id`.

In [ ]:
# Select a record set to preview records (first in list)
if len(record_sets) == 0:
    raise RuntimeError("No record sets available in dataset.")
record_set_id = record_sets[0]
print(f"\nSample records from record set {record_set_id}:")
count = 0
for record in dataset.records(record_set=record_set_id):
    print(record)
    count += 1
    if count >= 3:
        break

## 3. Data Extraction
Load data from *all* record sets into DataFrames for analysis. Reference record sets and field columns via their `@id`.

In [ ]:
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Show columns in the first record set
print(f"\nColumns in record set {record_set_id}:")
print(dataframes[record_set_id].columns.tolist())
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records, normalize numeric fields, group by key attributes. All fields referenced by their `@id`.

In [ ]:
df = dataframes[record_set_id]
print(f"Available fields (by @id):\n{list(df.columns)}")

# Choose a numeric field for analysis. If none exists, skip further EDA sections.
numeric_fields = df.select_dtypes(include='number').columns.tolist()
if not numeric_fields:
    print("No numeric fields available for EDA in this record set.")
else:
    numeric_field_id = numeric_fields[0]  # Pick the first numeric field
    print(f"\nSelected numeric field: {numeric_field_id}")
    threshold = df[numeric_field_id].mean()  # example threshold at mean
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records (where {numeric_field_id} > {threshold:.3f}): {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field if available
    categ_fields = df.select_dtypes(include='object').columns.tolist()
    group_field = None
    for f in categ_fields:
        # Heuristic: exclude id-like columns and require more than one unique value
        if f != numeric_field_id and df[f].nunique() > 1:
            group_field = f
            break
    if group_field:
        print(f"\nGroup by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we'll use matplotlib for basic plotting if numeric fields exist.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_fields) > 0:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()

## 6. Conclusion
We loaded the Mass Spectrometry Extracellular Vesicle dataset using `mlcroissant`, explored record sets and fields by `@id`, loaded data for each record set, applied basic filtering, normalization, and grouping for exploratory analysis, and visualized numeric distributions. Please refer to the official documentation or repository for more advanced usage and context.

**License:** [Open Data Commons Attribution License](https://opendatacommons.org/licenses/by/1-0/)
